# 05 -- Model Evaluation & SHAP Explainability

**Purpose:** Tune the best model (Random Forest) with GridSearchCV, evaluate on the test set, analyze residuals, and explain predictions with SHAP.

| Step | Description |
|---|---|
| 1 | Load data and best feature set |
| 2 | GridSearchCV tuning |
| 3 | Test set evaluation |
| 4 | Actual vs Predicted |
| 5 | Residual analysis |
| 6 | SHAP explainability |
| 7 | Save final model |

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import shap
import joblib

from src.config import (PROCESSED_DATA_PATH, IMAGES_DIR, REPORTS_DIR, MODELS_DIR,
                        TARGET_COL, TARGET_LOG_COL, RANDOM_STATE, TEST_SIZE)

pd.set_option('display.max_columns', None)

## 1. Load Data

In [2]:
df = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols = [c for c in df.columns if c not in [TARGET_COL, TARGET_LOG_COL]]

consensus_path = PROCESSED_DATA_PATH.parent / 'consensus_features.json'
if consensus_path.exists():
    with open(consensus_path) as fp:
        consensus_list = json.load(fp)
    consensus_list = [f for f in consensus_list if f in feature_cols]
else:
    consensus_list = feature_cols

X = df[consensus_list]
y = df[TARGET_LOG_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")

Train: (7307, 50)  Test: (1827, 50)


## 2. GridSearchCV Tuning -- Random Forest

In [3]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

rf = RandomForestRegressor(random_state=RANDOM_STATE)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid_search.fit(X_train_sc, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV R2:  {grid_search.best_score_:.4f}")
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 24 candidates, totalling 120 fits


Best params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Best CV R2:  0.9100


## 3. Test Set Evaluation

In [4]:
y_pred = best_model.predict(X_test_sc)

r2  = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("=== Best Model (Tuned Random Forest) ===")
print(f"R2:   {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

# Save final metrics
final_metrics = pd.DataFrame([{'Model': 'Tuned Random Forest',
                                'R2': round(r2, 4),
                                'RMSE': round(rmse, 4),
                                'MAE': round(mae, 4)}])
final_metrics.to_csv(REPORTS_DIR / 'final_model_metrics.csv', index=False)
print("Saved: final_model_metrics.csv")

=== Best Model (Tuned Random Forest) ===
R2:   0.9115
RMSE: 0.1980
MAE:  0.0913
Saved: final_model_metrics.csv


## 4. Actual vs Predicted

In [5]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_test, y_pred, alpha=0.35, color='royalblue', s=15, edgecolors='none')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual log(CLV)', fontsize=12)
ax.set_ylabel('Predicted log(CLV)', fontsize=12)
ax.set_title(f'Actual vs Predicted CLV  (R2={r2:.4f})', fontsize=13)
ax.legend()
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: actual_vs_predicted.png")

Saved: actual_vs_predicted.png


## 5. Residual Analysis

In [6]:
residuals = np.array(y_test) - np.array(y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuals, alpha=0.35, color='slategray', s=12)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted log(CLV)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residual Plot')

sns.histplot(residuals, kde=True, ax=axes[1], color='slategray')
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
fig.savefig(IMAGES_DIR / 'residuals.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: residuals.png")
print(f"\nResiduals -- Mean: {residuals.mean():.4f}  Std: {residuals.std():.4f}")

Saved: residuals.png

Residuals -- Mean: 0.0019  Std: 0.1979


## 6. SHAP Explainability

In [7]:
print("Computing SHAP values (TreeExplainer)...")
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_sc[:200])  # subset for speed

# Summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_sc[:200],
                  feature_names=consensus_list,
                  max_display=20,
                  show=False)
plt.title('SHAP Feature Importance (top 20)', fontsize=13)
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: shap_summary.png")

Computing SHAP values (TreeExplainer)...


Saved: shap_summary.png


## 7. Save Final Model

In [8]:
model_path = MODELS_DIR / 'best_random_forest.pkl'
joblib.dump(best_model, model_path)
print(f"Model saved -> {model_path}")

scaler_path = MODELS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"Scaler saved -> {scaler_path}")

Model saved -> D:\automation\Customer-Lifetime-Value-Prediction-For-AutoInsurance-Company\models\best_random_forest.pkl
Scaler saved -> D:\automation\Customer-Lifetime-Value-Prediction-For-AutoInsurance-Company\models\scaler.pkl


## Evaluation Summary

| Metric | Tuned RF | Reference Repo (RF) |
|---|---|---|
| R2 | ~0.91+ | 0.91 |
| RMSE | TBD (log scale) | 0.1956 |
| Improvements | + SHAP, + RFE, + XGBoost comparison | Baseline |

**Key insights from SHAP:**
- Monthly Premium Auto is the dominant predictor of CLV
- Number of Policies amplifies the premium signal
- Coverage type and Vehicle Class have significant non-linear effects

---
**Pipeline complete. Run `main.py` for end-to-end execution.**